# Time-to-Event Analysis of ALS Diagnosis
## UK Biobank (UKB) Cohort — Delta Protein Values

**Author:** Ximing Ran



## Executive Summary

This notebook investigates proteomic biomarkers associated with time to ALS diagnosis in the UK Biobank cohort. Using Cox proportional hazards regression on **delta protein values** (deviation from healthy control baseline), we identify proteins that predict time to diagnosis in ALS-affected individuals.

**Key Objectives:**
- Identify delta proteins significantly associated with ALS diagnosis timing
- Quantify hazard ratios for individual protein biomarkers
- Perform LASSO Cox regression for multi-protein feature selection

## UKB vs Miami Adaptations

| Aspect | Miami | UKB |
|--------|-------|-----|
| Groups | Phenoconverter + Pre-symptomatic | Phenoconverter + Pre-symptomatic
| Time variable | `-YrSinceOs` (Phenoconverter), `-YrSinceCen` (Pre-symptomatic) | `-YrSinceDi` (Phenoconverter), `-YrSinceCen` (Pre-symptomatic) |
| Event | Conversion to ALS | Confirmed ALS diagnosis |
| ID column | `UIDx` / `SampleID` | `eid` |
| Protein input | Delta matrix (wide) | Delta matrix (wide) |
| One obs/person | First visit | Baseline (already one per person) |


## 1. Load Libraries

In [ ]:
library(dplyr)
library(tidyr)
library(ggplot2)
library(ggrepel)
library(survival)
library(glmnet)
library(knitr)
library(kableExtra)
library(cowplot)
library(here)

set.seed(2025)
theme_set(theme_bw() + theme(legend.position = "bottom"))


## 2. Load Data

In [ ]:
# Delta protein matrix (wide format: eid x proteins)
protein_mat <- read.csv(here::here("data", "analysis_data", "ukb",
                                    "delta_matrix", "delta_matrix_wide.csv"))
# fill the NA with 0
protein_mat[is.na(protein_mat)] <- 0
# DE results from Miami Analysis 1 — defines protein_sign
protein_de <- read.csv("../../01-Differentially_Expressed_Proteins/Results/Mix_effect_model_lmer.csv")

all_proteins <- setdiff(colnames(protein_mat), "eid")

protein_sign <- protein_de %>%
  filter(significant == "Significant") %>%
  arrange(padj) %>%
  pull(Protein)

# Keep only proteins present in UKB delta matrix
protein_sign <- intersect(protein_sign, all_proteins)

cat("=== Protein Summary ===\n")
cat("Total proteins in delta matrix:   ", length(all_proteins), "\n")
cat("Significant DE proteins (Miami):  ", length(protein_sign), "\n")
cat("Overlapping proteins to analyse:  ", length(protein_sign), "\n")


### 2.1 Load Visit Info

In [ ]:
visit_info <- read.csv(here::here("data", "analysis_data", "ukb", "visit_info",
                                    "visit_info.csv"), row.names = 1)

visit_info <- visit_info %>%
  mutate(
    Group     = factor(Group, levels = c("Healthy control", "Pre-symptomatic", "Phenoconverter", "Pre-hospital", "Clinically manifest ALS")),
    GenoGroup = factor(GenoGroup),
    Sex       = factor(Sex, levels = c("Female", "Male"))
  )

cat("\n=== Group Counts ===\n")
print(table(visit_info$Group))


## 3. Prepare Survival Data

### Survival Time Construction

In UKB we have **Affected** individuals only (one baseline visit each).

- **Event** = 1 for all Affected individuals (confirmed ALS)
- **Time** = `YrSinceDi` (years since diagnosis at collection date)

> Note: `YrSinceDi` is time *from diagnosis*. A positive value means the sample
> was collected *after* diagnosis, negative means *before*. We use the absolute
> value so that time is always positive (time from collection to diagnosis for
> pre-diagnosis samples, or time from diagnosis to collection for post-diagnosis).
> Individuals with `YrSinceDi = 0` or `NA` are excluded.

### Statistical Model

$$h(t|X) = h_0(t) \exp(\beta_1 \cdot \text{Protein} + \beta_2 \cdot \text{Age} + \beta_3 \cdot \text{Sex} + \beta_4 \cdot \text{Genotype})$$


In [ ]:
# Keep Affected individuals with valid YrSinceDi
vis_sub <- visit_info %>%
  filter(Group %in% c("Pre-symptomatic", "Phenoconverter")) %>%
  mutate(
    event   = ifelse(Group != "Pre-symptomatic", 1L, 0L),
    time_yr = ifelse(Group == "Pre-symptomatic", abs(as.numeric(YrSinceCen)), abs(as.numeric(YrSinceDi + 2)))
  ) %>%
  filter(is.finite(time_yr), time_yr > 0)

# Relevel GenoGroup so a common reference is first
vis_sub$GenoGroup <- relevel(droplevels(vis_sub$GenoGroup), ref = "None identified")

cat("=== Survival Data Summary ===\n")
cat("Total individuals:", nrow(vis_sub), "\n")
cat("Events (Affected):", sum(vis_sub$event), "\n")
cat("time_yr range:    ", round(min(vis_sub$time_yr), 2),
    "to", round(max(vis_sub$time_yr), 2), "yr\n")
cat("\nGenoGroup distribution:\n")
print(table(vis_sub$GenoGroup))


In [ ]:
# Merge with delta protein matrix
dat0 <- vis_sub %>%
  left_join(protein_mat, by = "eid")

# Covariates
covar <- c("CollAge", "Sex", "GenoGroup")

cat("\nMerged data dimensions:", dim(dat0), "\n")
cat("Complete cases (all proteins + covariates):", 
    sum(complete.cases(dat0[, c("time_yr", "event", covar)])), "\n")


## 4. Univariate Cox Regression

In [ ]:
results_list <- vector("list", length(protein_sign))

for (i in seq_along(protein_sign)) {

  protein_name <- protein_sign[i]


  tryCatch({
    dat_protein <- dat0 %>%
      select(eid, time_yr, event, all_of(protein_name), all_of(covar)) %>%
      drop_na()

    if (nrow(dat_protein) < 5 || sum(dat_protein$event) < 3) {
      results_list[[i]] <- data.frame(
        Protein = protein_name, N = nrow(dat_protein),
        N_events = sum(dat_protein$event), coef = NA, exp_coef = NA,
        se_coef = NA, z = NA, p_value = NA, Status = "Insufficient data"
      )
      return(invisible(NULL))
    }

    formula_str <- paste0("Surv(time_yr, event) ~ ", protein_name,
                          " + ", paste(covar, collapse = " + "))
    fit <- coxph(as.formula(formula_str), data = dat_protein)

    cs <- summary(fit)$coefficients
    pc <- cs[1, , drop = FALSE]

    results_list[[i]] <- data.frame(
      Protein  = protein_name,
      N        = nrow(dat_protein),
      N_events = sum(dat_protein$event),
      coef     = pc[1, "coef"],
      exp_coef = pc[1, "exp(coef)"],
      se_coef  = pc[1, "se(coef)"],
      z        = pc[1, "z"],
      p_value  = pc[1, "Pr(>|z|)"],
      Status   = "Success"
    )

  }, error = function(e) {
    results_list[[i]] <<- data.frame(
      Protein = protein_name, N = NA, N_events = NA,
      coef = NA, exp_coef = NA, se_coef = NA, z = NA, p_value = NA,
      Status = paste("Error:", e$message)
    )
  })
}

cox_results <- bind_rows(results_list) %>%
  mutate(fdr = ifelse(!is.na(p_value), p.adjust(p_value, method = "BH"), NA)) %>%
  arrange(p_value)

cat("\nDone! Proteins fitted:", sum(cox_results$Status == "Success"), "\n")


## 5. Volcano Plot

In [ ]:
volcano_df <- cox_results %>%
  filter(!is.na(p_value), !is.na(coef)) %>%
  mutate(
    negLogP      = -log10(p_value),
    significance = case_when(
      fdr < 0.05 & coef > 0 ~ "UP",
      fdr < 0.05 & coef < 0 ~ "DOWN",
      TRUE                   ~ "NO"
    )
  )

# BH threshold for horizontal line
bh_threshold <- function(p_values, alpha = 0.05) {
  m        <- length(p_values)
  p_sorted <- sort(p_values)
  k_values <- which(p_sorted <= (1:m) / m * alpha)
  if (length(k_values) == 0) return(alpha / m)
  p_sorted[max(k_values)]
}
p_threshold <- bh_threshold(volcano_df$p_value)

n_up   <- sum(volcano_df$significance == "UP")
n_down <- sum(volcano_df$significance == "DOWN")

top_up   <- volcano_df %>% filter(significance == "UP")   %>% arrange(p_value) %>% slice_head(n = 10)
top_down <- volcano_df %>% filter(significance == "DOWN")  %>% arrange(p_value) %>% slice_head(n = 10)
top_label <- bind_rows(top_up, top_down)

y_max_data <- max(volcano_df$negLogP, na.rm = TRUE)
y_max_plot <- y_max_data * 1.3
text_y     <- y_max_data * 1.15

options(repr.plot.width = 10, repr.plot.height = 7)

p_volcano <- ggplot(volcano_df, aes(x = coef, y = negLogP, color = significance)) +
  geom_point(alpha = 0.6, size = 2) +
  geom_hline(yintercept = -log10(p_threshold),
             linetype = "dashed", color = "gray40", linewidth = 0.8) +
  annotate("text",
           x = min(volcano_df$coef, na.rm = TRUE) + 0.05, y = text_y,
           label = paste0("Protective: ", n_down),
           hjust = 0, size = 5, color = "#427ebf", fontface = "bold") +
  annotate("text",
           x = max(volcano_df$coef, na.rm = TRUE) - 0.05, y = text_y,
           label = paste0("Risk-Associated: ", n_up),
           hjust = 1, size = 5, color = "#A70C20", fontface = "bold") +
  scale_color_manual(
    values = c(UP = "#A70C20", DOWN = "#427ebf", NO = "grey60"),
    labels = c(UP = "Increased Risk (FDR<0.05)", DOWN = "Decreased Risk (FDR<0.05)",
               NO = "Not Significant")
  ) +
  scale_y_continuous(limits = c(0, y_max_plot), expand = c(0, 0)) +
  labs(
    x        = "Cox Coefficient (log Hazard Ratio)",
    y        = expression(-log[10]("P-value")),
    title    = "Proteomic Biomarkers of ALS Diagnosis Time (UKB)",
    subtitle = paste0("FDR<0.05 threshold: p = ",
                      formatC(p_threshold, format = "e", digits = 2)),
    color    = "Association"
  ) +
  theme_classic(base_size = 14) +
  theme(
    legend.position  = "bottom",
    plot.title       = element_text(hjust = 0.5, face = "bold", size = 16),
    plot.subtitle    = element_text(hjust = 0.5, size = 11, color = "gray40"),
    axis.title       = element_text(face = "bold"),
    panel.grid.major = element_line(color = "gray90", linewidth = 0.3),
    panel.grid.minor = element_blank()
  ) +
  geom_label_repel(
    data = top_label,
    aes(label = Protein, color = significance),
    size = 3.5, max.overlaps = 50, min.segment.length = 0,
    show.legend = FALSE, box.padding = 0.5, point.padding = 0.3,
    ylim = c(NA, y_max_plot), fontface = "bold"
  )

print(p_volcano)


## 6. Results Table

In [ ]:
cox_results_table <- volcano_df %>%
  select(Protein, N, N_events, coef, exp_coef, p_value, fdr, significance) %>%
  arrange(p_value) %>%
  mutate(
    coef     = round(coef,     3),
    exp_coef = round(exp_coef, 3),
    p_value  = round(p_value,  4),
    fdr      = round(fdr,      4)
  )

outdir <- file.path("Results", "1.Cox_Model_Baseline_UKB")
dir.create(outdir, recursive = TRUE, showWarnings = FALSE)

write.csv(cox_results_table,
          file.path(outdir, "cox_results_univariate_ukb.csv"),
          row.names = FALSE)

cat("Univariate results saved.\n")
cat("Significant proteins (FDR<0.05):", sum(cox_results_table$significance != "NO"), "\n")

knitr::kable(cox_results_table,
             format    = "pipe",
             caption   = "Cox Regression Results — UKB Delta Proteins",
             col.names = c("Protein", "N", "Events", "Coefficient",
                           "Hazard Ratio", "P-value", "FDR", "Significance"),
             align     = c("l", "r", "r", "r", "r", "r", "r", "c"))


## 7. LASSO Cox Regression

### Model

$$\min_{\beta} \left\{-\ell(\beta) + \lambda \sum_{j=1}^{p} |\beta_j|\right\}$$

- Only protein coefficients are penalised (`penalty.factor = 1`)
- Covariates (Age, Sex, GenoGroup) are **unpenalised** (`penalty.factor = 0`)
- Lambda selected by leave-one-out cross-validation (LOOCV)


### 7.1 Prepare LASSO Input

In [ ]:
dat_lasso <- dat0 %>%
  select(eid, time_yr, event,
         all_of(protein_sign), all_of(covar)) %>%
  drop_na()

cat("=== LASSO Data Summary ===\n")
cat("Total individuals:", nrow(dat_lasso), "\n")
cat("Events (Affected):", sum(dat_lasso$event), "\n")
cat("Total proteins:   ", length(protein_sign), "\n")

# ── Protein matrix ────────────────────────────────────────────────────────────
X_proteins <- as.matrix(dat_lasso[, protein_sign])

# ── Covariates ────────────────────────────────────────────────────────────────
X_sex  <- as.numeric(dat_lasso$Sex == "Male")
X_age  <- dat_lasso$CollAge
X_geno <- model.matrix(~ GenoGroup - 1, data = dat_lasso)
colnames(X_geno) <- paste0("Geno_", gsub("GenoGroup", "", colnames(X_geno)))

# Drop reference level (first column)
X_geno <- X_geno[, -1, drop = FALSE]

X_covariates <- cbind(Age = X_age, Sex_Male = X_sex, X_geno)
X_full       <- cbind(X_proteins, X_covariates)

# Penalty factors: 1 for proteins, 0 for covariates
penalty_factors <- c(rep(1, ncol(X_proteins)), rep(0, ncol(X_covariates)))

# Survival object
y_surv <- Surv(dat_lasso$time_yr, dat_lasso$event)

cat("\nDesign matrix dimensions:", dim(X_full), "\n")
cat("Penalised features:       ", sum(penalty_factors == 1), "\n")
cat("Unpenalised features:     ", sum(penalty_factors == 0), "\n")


### 7.2 Cross-Validation

In [ ]:
cat("Running LOOCV (nfolds =", nrow(X_full), ")... this may take a few minutes.\n")

set.seed(2025)
# suppress the warning
suppressWarnings(
cv_fit <- cv.glmnet(
  x              = X_full,
  y              = y_surv,
  family         = "cox",
  penalty.factor = penalty_factors,
  nfolds         = nrow(X_full),   # LOOCV
  type.measure   = "deviance",
  alpha          = 1               # LASSO
))

lambda_min <- cv_fit$lambda.min
lambda_1se <- cv_fit$lambda.1se

cat("\nOptimal Lambda Selection:\n")
cat("  Lambda Min (min CV error):", formatC(lambda_min, format = "e", digits = 3), "\n")
cat("  Lambda 1SE (parsimony):   ", formatC(lambda_1se, format = "e", digits = 3), "\n")
cat("  CV deviance at Lambda Min:", round(min(cv_fit$cvm), 4), "\n")


In [ ]:
options(repr.plot.width = 12, repr.plot.height = 5)
par(mfrow = c(1, 2), mar = c(4.5, 4.5, 3, 1))

plot(cv_fit, main = "LASSO Cox: CV Curve (UKB)")
abline(v = log(lambda_min), col = "red",  lty = 2, lwd = 2)
abline(v = log(lambda_1se), col = "blue", lty = 2, lwd = 2)
legend("topleft", legend = c("Lambda Min", "Lambda 1SE"),
       col = c("red", "blue"), lty = 2, lwd = 2, bty = "n")

plot(cv_fit$glmnet.fit, xvar = "lambda", main = "LASSO Coefficient Paths (UKB)")
abline(v = log(lambda_min), col = "red",  lty = 2, lwd = 2)
abline(v = log(lambda_1se), col = "blue", lty = 2, lwd = 2)

par(mfrow = c(1, 1))


### 7.3 Selected Proteins

In [ ]:
# ── Lambda Min ────────────────────────────────────────────────────────────────
fit_min    <- glmnet(X_full, y_surv, family = "cox",
                     penalty.factor = penalty_factors, lambda = lambda_min, alpha = 1)
coef_min   <- coef(fit_min, s = lambda_min)
nz_idx_min <- which(coef_min != 0)
sel_min    <- rownames(coef_min)[nz_idx_min]
coef_min_v <- as.vector(coef_min[nz_idx_min])

prot_mask_min      <- sel_min %in% protein_sign
sel_proteins_min   <- sel_min[prot_mask_min]
sel_coefs_min      <- coef_min_v[prot_mask_min]

# ── Lambda 1SE ────────────────────────────────────────────────────────────────
fit_1se    <- glmnet(X_full, y_surv, family = "cox",
                     penalty.factor = penalty_factors, lambda = lambda_1se, alpha = 1)
coef_1se   <- coef(fit_1se, s = lambda_1se)
nz_idx_1se <- which(coef_1se != 0)
sel_1se    <- rownames(coef_1se)[nz_idx_1se]
coef_1se_v <- as.vector(coef_1se[nz_idx_1se])

prot_mask_1se      <- sel_1se %in% protein_sign
sel_proteins_1se   <- sel_1se[prot_mask_1se]
sel_coefs_1se      <- coef_1se_v[prot_mask_1se]

cat("=== LASSO Feature Selection Results ===\n\n")
cat("Lambda Min Model:\n")
cat("  Selected proteins:", length(sel_proteins_min), "of", length(protein_sign), "\n")
cat("  Reduction:        ",
    round(100 * (1 - length(sel_proteins_min) / length(protein_sign)), 1), "%\n\n")
cat("Lambda 1SE Model:\n")
cat("  Selected proteins:", length(sel_proteins_1se), "of", length(protein_sign), "\n")
cat("  Reduction:        ",
    round(100 * (1 - length(sel_proteins_1se) / length(protein_sign)), 1), "%\n")


In [ ]:
# Helper: build LASSO results table with univariate comparison
make_lasso_table <- function(sel_proteins, sel_coefs, lambda_val, label) {
  if (length(sel_proteins) == 0) {
    cat("No proteins selected at", label, "\n"); return(invisible(NULL))
  }
  tbl <- data.frame(Protein = sel_proteins, LASSO_Coef = sel_coefs,
                    LASSO_HR = exp(sel_coefs)) %>%
    arrange(desc(abs(LASSO_Coef))) %>%
    left_join(cox_results %>% select(Protein, coef, exp_coef, p_value, fdr),
              by = "Protein") %>%
    rename(Univariate_Coef = coef, Univariate_HR = exp_coef,
           Univariate_P = p_value, Univariate_FDR = fdr) %>%
    mutate(across(where(is.numeric), ~ round(.x, 4)))

  print(knitr::kable(tbl, format = "pipe",
    caption = paste0("Proteins Selected by LASSO (", label, ", lambda = ",
                     formatC(lambda_val, format = "e", digits = 2), ")"),
    align = c("l", rep("r", 6))))

  write.csv(tbl, file.path(outdir, paste0("lasso_results_", label, "_ukb.csv")),
            row.names = FALSE)
  invisible(tbl)
}

lasso_min_tbl <- make_lasso_table(sel_proteins_min, sel_coefs_min, lambda_min, "lambda_min")
cat("\n")
lasso_1se_tbl <- make_lasso_table(sel_proteins_1se, sel_coefs_1se, lambda_1se, "lambda_1se")


## 8. Univariate vs LASSO Comparison Plot

In [ ]:
if (length(sel_proteins_min) > 0) {

  options(repr.plot.width = 10, repr.plot.height = 6)

  comparison_df <- lasso_min_tbl %>%
    select(Protein, LASSO_Coef, Univariate_Coef) %>%
    pivot_longer(cols = c(LASSO_Coef, Univariate_Coef),
                 names_to = "Method", values_to = "logHR") %>%
    mutate(
      Method = factor(Method,
                      levels = c("Univariate_Coef", "LASSO_Coef"),
                      labels = c("Univariate", "LASSO")),
      HR = exp(logHR)
    )

  protein_order <- comparison_df %>%
    filter(Method == "Univariate") %>%
    arrange(HR) %>%
    pull(Protein)

  comparison_df <- comparison_df %>%
    mutate(Protein = factor(Protein, levels = protein_order))

  p_comp <- ggplot(comparison_df, aes(x = Protein, y = HR, fill = Method)) +
    geom_bar(stat = "identity", position = "dodge", alpha = 0.8) +
    geom_hline(yintercept = 1, linetype = "solid", color = "black") +
    coord_flip() +
    scale_fill_manual(values = c(Univariate = "#8856a7", LASSO = "#43a2ca")) +
    labs(
      title    = "Univariate vs LASSO Cox Hazard Ratios (UKB)",
      subtitle = "LASSO HRs typically shrink toward 1",
      x = "Protein", y = "Hazard Ratio (HR)", fill = "Method"
    ) +
    theme_classic(base_size = 12) +
    theme(plot.title    = element_text(face = "bold", hjust = 0.5),
          plot.subtitle = element_text(hjust = 0.5, color = "gray40"),
          axis.text.y   = element_text(face = "bold"),
          legend.position = "bottom")

  print(p_comp)

} else {
  cat("No proteins selected at lambda.min — skipping comparison plot.\n")
}


## 9. Output Files

| File | Description |
|------|-------------|
| `cox_results_univariate_ukb.csv` | Univariate Cox results for all proteins |
| `lasso_results_lambda_min_ukb.csv` | LASSO selected proteins at lambda.min |
| `lasso_results_lambda_1se_ukb.csv` | LASSO selected proteins at lambda.1se |
